In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import numpy as np
import torch

from loka.models import BreakthroughModel
from loka.data import load_breakthrough_states

import os
from dotenv import load_dotenv
load_dotenv()

SEED = 42
torch.manual_seed(SEED)

BASE_DIR = os.getenv("BASE_DIR")

data_dir = f"{BASE_DIR}/data/game_states/e.txt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

X_np,Y_np = load_breakthrough_states(data_dir)
X,Y = torch.Tensor(X_np), torch.Tensor(Y_np)

In [ ]:
X = X.reshape(-1, 2, 5, 5)                           # (N, 2, 5, 5)
ones = torch.ones(X.shape[0], 1, 5, 5, dtype=X.dtype, device=X.device)
X = torch.cat([X, ones], dim=1)   
X

In [ ]:
my_resnet = resnet18(weights=ResNet18_Weights.DEFAULT, progress=False)

In [ ]:
my_resnet(X[:2])

In [ ]:
class ResNet18(BreakthroughModel):
    def __init__(self):
        super().__init__()
        self.resnet18 = resnet18(weights=ResNet18_Weights.DEFAULT, progress=False)
        self.resnet18 .fc = nn.Linear(my_resnet.fc.in_features, 1) # linear layer finally

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            y_pred = self.resnet18(x)
            return y_pred
    
    

In [ ]:
res = ResNet18()

In [ ]:
res(X[:2])